In [2]:
import pandas as pd

df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

In [3]:
df_agent

,question,answer_agent,answer_orig,tool_calls,cost,document
0,Can I take this course at my own pace and stil...,"No — for this course, you can follow it in sel...","No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001202,69d122f12e
1,Is a certificate available if I complete the c...,No — certificates are only available if you co...,"No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001157,69d122f12e
2,Do self-paced learners get any certificate for...,No — self-paced learners do not get a certific...,"No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001120,69d122f12e
3,Why are certificates not issued for the self-p...,Certificates aren’t issued for the self-paced ...,"No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001275,69d122f12e
4,Is peer review of capstone projects required i...,Yes — peer review of capstone projects is requ...,"No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001331,69d122f12e
5,How do students join the Office Hours or live ...,Students don’t use the Zoom link directly. The...,The zoom link is only published to instructors...,"[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001634,489dd1c9d9
6,Where should I look for the live video URL bef...,"Before an Office Hours session starts, look fo...",The zoom link is only published to instructors...,"[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001620,489dd1c9d9
7,"Can I watch the workshop live on YouTube, and ...",Yes — you can watch the workshop live on YouTu...,The zoom link is only published to instructors...,"[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001465,489dd1c9d9
8,How are questions supposed to be submitted dur...,Questions during the live session should be su...,The zoom link is only published to instructors...,"[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001446,489dd1c9d9
9,Why shouldn’t I ask questions in the chat duri...,You shouldn’t ask questions in the chat during...,The zoom link is only published to instructors...,"[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001433,489dd1c9d9


In [4]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [5]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [8]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

token = os.getenv("GEMINI_API_KEY")
endpoint = "https://generativelanguage.googleapis.com/v1beta/openai/"
model='gemini-2.5-flash'

openai_client = OpenAI(
    base_url=endpoint,
    api_key=token,
)

In [11]:
import json
from evaluation_utils import calc_total_price, llm_structured

def evaluate_agent_answer(rec, model="gemini-2.5-flash"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [12]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning="The agent's answer accurately reflects the ground truth. It correctly states that a certificate is not awarded for self-paced mode and explains that it requires participation in a 'live cohort' due to the capstone peer-review process being available only while the course is running. This covers all key information from the original answer.", answer_score='good', trajectory_reasoning="The agent made a single, highly relevant search query that included keywords from the question such as 'self-paced', 'certificate', 'own pace', and 'FAQ'. This query was appropriate for finding the necessary information to answer the question, and only one call was needed.", trajectory_score='good')

In [13]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage